In [0]:
%run ./config

In [0]:
SECRET_SCOPE = get_widget_param("secret_scope", "eventhub")
SECRET_KEY = get_widget_param("secret_key", "eh-connection-string")
EH_NAMESPACE = get_widget_param("eh_namespace", "evhua5816bd")
EH_NAME = get_widget_param("eh_name", "roksolana-wikipedia-recentchange")
CATALOG = get_widget_param("catalog", "dbr_dev_ua5816bd")
SCHEMA_LANDING = get_widget_param("schema_landing", "roksolana_shendiu770")
SCHEMA_BRONZE = get_widget_param("schema_bronze", "roksolana_shendiu770_bronze")
TARGET_TABLE_NAME = get_widget_param("target_table_name", "wikipedia_recentchange_bronze")
CHECKPOINT_SUBDIR = get_widget_param("checkpoint_subdir", "wikipedia_recentchange")
STARTING_OFFSETS = get_widget_param("starting_offsets", "earliest")
MAX_OFFSETS_PER_TRIGGER = get_widget_param("max_offsets_per_trigger", "50000")

KAFKA_REQUEST_TIMEOUT_MS = "60000"
KAFKA_SESSION_TIMEOUT_MS = "30000"
FAIL_ON_DATA_LOSS = "false"

required_params = {
    "eh_namespace": EH_NAMESPACE,
    "catalog": CATALOG,
    "schema_landing": SCHEMA_LANDING,
    "schema_bronze": SCHEMA_BRONZE,
}
missing = [name for name, value in required_params.items() if not value]
if missing:
    raise ValueError(f"Missing required parameters: {', '.join(missing)}")

CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/{CHECKPOINT_SUBDIR}"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.{TARGET_TABLE_NAME}"

EH_CONN_STR = get_eventhub_connection_string(SECRET_SCOPE, SECRET_KEY)

KAFKA_OPTIONS = build_kafka_options(
    EH_NAMESPACE, EH_NAME, EH_CONN_STR,
    KAFKA_REQUEST_TIMEOUT_MS, KAFKA_SESSION_TIMEOUT_MS,
    MAX_OFFSETS_PER_TRIGGER, FAIL_ON_DATA_LOSS, STARTING_OFFSETS
)

In [0]:
import logging
from pyspark.sql.functions import col, current_timestamp

logger = logging.getLogger("wikipedia_eventhub_consumer")
logger.setLevel(logging.INFO)

raw_stream = (
    spark.readStream
    .format("kafka")
    .options(**KAFKA_OPTIONS)
    .load()
)

bronze_stream = (
    raw_stream
    .select(
        col("value").cast("string").alias("raw_payload"),
        col("key").cast("string").alias("kafka_key"),
        col("topic").alias("eh_name"),
        col("partition").alias("eh_partition"),
        col("offset").alias("eh_offset"),
        col("timestamp").alias("eh_enqueued_timestamp"),
        current_timestamp().alias("etl_processed_timestamp"),
    )
)

logger.info("Starting Event Hub consumer (availableNow), target_table=%s", TARGET_TABLE)

query = (
    bronze_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

logger.info("Consumer finished.")